# SAM-GA Optimization: Genetic Algorithm for Sensor Design

## Overview
This notebook demonstrates the use of Genetic Algorithm (GA) optimization for designing infrared microbolometer sensors. The goal is to find optimal basis function parameters (Gaussian means and standard deviations) that maximize the separability between different substances using Spectral Angle Mapper (SAM) distance metrics.

## Key Concepts
- **Basis Functions**: Gaussian curves defined by mean (µ) and standard deviation (σ) parameters
- **Chromosome Encoding**: Each chromosome represents 4 basis functions as [µ1, σ1, µ2, σ2, µ3, σ3, µ4, σ4]
- **Fitness Function**: Based on min-based dissimilarity score of the distance matrix between substances
- **Distance Metric**: Spectral Angle Mapper (SAM) for measuring spectral similarity
- **Optimization Target**: Maximize separability between white powder substances

## Objectives
1. Load spectral data for white powder substances
2. Set up GA parameters and fitness function
3. Run optimization to find optimal basis function parameters
4. Analyze results and visualize the best sensor designs
5. Perform diversity analysis of the final population

Notes:
- This notebook follows the standardized structure and documentation style
- Uses imports from the src package where possible
- Serves as a clean, professional reference for future experiment notebooks

## 1. Data Loading and Preprocessing

We begin by loading the spectral data for white powder substances and setting up the necessary parameters for sensor simulation and GA optimization.

In [ ]:
# Imports and Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pygad
from tqdm import tqdm
import warnings

warnings.filterwarnings("ignore")

# Import from the new package structure
from lwi_microbolometer_design import (
    simulate_sensor_output,
    compute_distance_matrix,
    spectral_angle_mapper,
    min_based_dissimilarity_score,
    generate_basis_from_chromosome,
    distance_optimal_pairing,
    generate_distance_matrix,
    vat_reorder,
    visualize_distance_matrix_large,
)

# Plotting style
plt.style.use("default")
plt.rcParams["figure.figsize"] = (12, 8)
plt.rcParams["font.size"] = 12

print("✅ Imports and setup completed successfully")

In [ ]:
# Load substances and generate basis functions
# --------------------------------------------------------------------------------------------------

# File paths for input data
spectral_data_file = "../data/Test 3 - 4 White Powers/white_powders_with_labels.xlsx"
air_transmittance_file = "../data/Test 3 - 4 White Powers/Air transmittance.xlsx"

# Configurable Parameters
atmospheric_distance_ratio = (
    0.11  # Used to model the effect of atmospheric conditions on measurements
)
temperature_K = 293.15  # Temperature in Kelvin
air_refractive_index = 1  # Refractive index of air

# Load the spectral data of substances
substances_spectral_data = pd.read_excel(spectral_data_file)

# Extract the wavelength values (first column)
wavelengths = substances_spectral_data.iloc[:, :1].to_numpy()  # Shape = (d, 1)

# Extract substance labels (column names excluding the first)
substance_names = substances_spectral_data.columns[1:].to_numpy()

# Extract emissivity curves (all data except the first column)
emissivity_curves = substances_spectral_data.iloc[:, 1:].to_numpy()

# Load air transmittance matrix
air_transmittance = np.array(pd.read_excel(air_transmittance_file, header=None))
air_transmittance = air_transmittance[:, 1:]  # Remove the spectra column

print(f"✅ Loaded spectral data for {len(substance_names)} substances")
print(f"📊 Wavelength range: {wavelengths[0][0]:.1f} - {wavelengths[-1][0]:.1f} µm")
print(f"📊 Number of wavelength points: {len(wavelengths)}")

# Visualization: Spectral Emissivity Curves
plt.figure(figsize=(10, 6))
for name in substance_names:
    plt.plot(wavelengths, substances_spectral_data[name], label=name, linewidth=2)

plt.xlabel("Wavelength (µm)")
plt.ylabel("Emissivity Value")
plt.title("Spectral Emissivity Curves of White Powder Substances")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Basis Function Generation
# --------------------------------------------------------------------------------------------------
# Note: generate_basis_from_chromosome is now imported from lwi_microbolometer_design.optimization
# This function translates GA chromosomes into Gaussian basis function curves with peak alignment.

print("✅ Basis function generation function available from src package")

In [ ]:
# # Sensor Output Simulation with Variability
# # ==================================================================

# def generate_variable_sensor_outputs(
#     wavelengths,
#     emissivity_curve,
#     basis_functions,
#     n_samples=5,
#     temperature_range=(280, 320),      # K
#     distance_range=(0.8, 1.2),         # relative scaling
#     noise_std=0.01
# ):
#     """
#     Generate multiple sensor outputs for one substance under variability.
#     """
#     outputs = []
#     for _ in range(n_samples):
#         temp = np.random.uniform(*temperature_range)
#         dist = np.random.uniform(*distance_range)

#         sensor_output = simulate_sensor_output(
#             wavelengths=wavelengths,
#             substances_emissivity=[emissivity_curve],
#             basis_functions=basis_functions,
#             temperature_K=temp,
#             atmospheric_distance_ratio=dist,
#             air_refractive_index=air_refractive_index,
#             air_transmittance=air_transmittance
#         )[0]  # single substance, take first output

#         # Add measurement noise
#         noisy_output = sensor_output + np.random.normal(0, noise_std, size=sensor_output.shape)
#         outputs.append(noisy_output)

#     return np.array(outputs)

#     # # Generate all random variables at once
#     # temps = np.random.uniform(*temperature_range, size=n_samples)
#     # dists = np.random.uniform(*distance_range, size=n_samples)

#     # # Assumes simulate_sensor_output is modified to handle arrays of temps and dists
#     # # and returns a shape of (n_samples, n_features)
#     # sensor_outputs = simulate_sensor_output_vectorized(
#     #     wavelengths=wavelengths,
#     #     substances_emissivity=[emissivity_curve],
#     #     basis_functions=basis_functions,
#     #     temperatures_K=temps, # Pass arrays
#     #     atmospheric_distance_ratios=dists, # Pass arrays
#     #     # ... other args ...
#     # )[0]

#     # # Add noise to all samples at once
#     # noise = np.random.normal(0, noise_std, size=sensor_outputs.shape)
#     # return sensor_outputs + noise

In [ ]:
# def compute_group_distance_matrix(groups, separability_func):
#     """
#     Computes a group-to-group separability matrix using a given separability function.

#     Parameters:
#     - groups (list of 2D arrays): Each element is (m, k), where columns are signals for one substance.
#     - separability_func (function): A function separability_func(groupA, groupB) -> float.

#     Returns:
#     - matrix (2D array): Separability matrix (n x n), symmetric, with zeros on the diagonal.
#     """
#     n = len(groups)
#     # print(n, "groups in compute_group_distance_matrix")
#     matrix = np.zeros((n, n))

#     for i in range(n):
#         for j in range(i + 1, n):
#             sep_value = separability_func(groups[i], groups[j])
#             matrix[i, j] = sep_value
#             matrix[j, i] = sep_value

#     return matrix

In [ ]:
# # Group Based Separability Metrics
# # ==================================================================


# def separability_min(groupA, groupB):
#     dists = []
#     for sigA in groupA.T:
#         for sigB in groupB.T:
#             dists.append(spectral_angle_mapper(sigA, sigB))
#     return np.min(dists)


# def separability_mean(groupA, groupB):
#     dists = []
#     for sigA in groupA.T:
#         for sigB in groupB.T:
#             dists.append(spectral_angle_mapper(sigA, sigB))
#     return np.mean(dists)


# def separability_mean_over_intra(groupA, groupB):
#     # inter distances
#     inter_dists = []
#     for sigA in groupA.T:
#         for sigB in groupB.T:
#             inter_dists.append(spectral_angle_mapper(sigA, sigB))
#     inter_mean = np.mean(inter_dists)

#     # intra variability (average within each group)
#     def intra_variability(group):
#         d = []
#         k = group.shape[1]
#         for i in range(k):
#             for j in range(i + 1, k):
#                 d.append(spectral_angle_mapper(group[:, i], group[:, j]))
#         return np.mean(d) if d else 0.0

#     intraA = intra_variability(groupA)
#     intraB = intra_variability(groupB)

#     return inter_mean / (1 + (intraA + intraB) / 2.0)


# def separability_mean_distance_over_variance(groupA, groupB):
#     """
#     Separation = distance(meanA, meanB) / (varA + varB)
#     """
#     # group mean (vector)
#     meanA = np.mean(groupA, axis=1)
#     meanB = np.mean(groupB, axis=1)

#     # inter-group distance
#     inter = spectral_angle_mapper(meanA, meanB)

#     # intra-group variance (average distance to mean)
#     def intra_variance(group, mean_vec):
#         d = [spectral_angle_mapper(sig, mean_vec) for sig in group.T]
#         return np.mean(d)

#     varA = intra_variance(groupA, meanA)
#     varB = intra_variance(groupB, meanB)

#     return inter / (1 + (varA + varB))

In [ ]:
# # Fitness Function Factory for Group-Based Separability
# # ==================================================================
# import time
# def make_fitness_func_group(separability_func):
#     """
#     Factory to create a fitness function compatible with PyGAD,
#     with a specific separability_func baked in.
#     """

#     def fitness_func_group(ga_instance, chromosome, chromosome_idx):
#         start = time.time()
#         # Step 1: Generate basis functions
#         current_basis_funcs = generate_basis_from_chromosome(chromosome, wavelengths)

#         # Step 2: Generate multiple sensor outputs per substance
#         groups = []
#         print('emiss shape', emissivity_curves.shape)
#         for emissivity_curve in emissivity_curves.T:
#             outputs = generate_variable_sensor_outputs(
#                 wavelengths=wavelengths,
#                 emissivity_curve=emissivity_curve,
#                 basis_functions=current_basis_funcs,
#                 n_samples=5,                 # configurable
#                 temperature_range=(280, 320),
#                 distance_range=(0.8, 1.2),
#                 noise_std=0.01
#             )
#             print('outputs shape', outputs.shape)
#             # outputs: shape (num_samples, m) → transpose to (m, num_samples)
#             groups.append(outputs)


#         # Step 3: Compute group-to-group separability matrix
#         separability_matrix = compute_group_distance_matrix(
#             groups,
#             separability_func=separability_func
#         )

#         # Step 4: Calculate fitness from the separability matrix
#         fitness = min_based_dissimilarity_score(separability_matrix)
#         end = time.time()
#         print("Runtime (seconds):", end - start)
#         return fitness

#     return fitness_func_group

In [ ]:
# Chromosome Distance Calculation
# --------------------------------------------------------------------------------------------------
# Note: distance_optimal_pairing is now imported from lwi_microbolometer_design.optimization
# This function computes the distance between two chromosomes using optimal pairing of basis functions.

print("✅ Chromosome distance calculation function available from src package")

## 2. Genetic Algorithm Setup

We define the fitness function and GA parameters for optimizing the basis function parameters.


In [ ]:
# Fitness Function based on Min-Based Dissimilarity Score
# --------------------------------------------------------------------------------------------------


def fitness_func(ga_instance, chromosome, chromosome_idx):
    """
    Calculates the fitness of a given chromosome of µ and σ parameters.
    The fitness is determined by the min-based dissimilarity score.

    This function is specific to this experiment's GA configuration and remains
    in the notebook as it requires access to local variables (wavelengths,
    emissivity_curves, etc.).
    """
    # The 'chromosome' is now a list of parameters [µ1, σ1, µ2, σ2, ...]
    # Step 1: Generate the basis function curves directly from the chromosome.
    current_basis_funcs = generate_basis_from_chromosome(chromosome, wavelengths)

    # Step 2: Simulate the sensor output using the generated basis functions
    sensor_outputs = simulate_sensor_output(
        wavelengths=wavelengths,
        substances_emissivity=emissivity_curves,
        basis_functions=current_basis_funcs,  # Use the newly generated functions
        temperature_K=temperature_K,
        atmospheric_distance_ratio=atmospheric_distance_ratio,
        air_refractive_index=air_refractive_index,
        air_transmittance=air_transmittance,
    )

    # Step 3: Compute the distance matrix from the sensor outputs
    distance_matrix = compute_distance_matrix(sensor_outputs, spectral_angle_mapper)

    # Step 4: Calculate the dissimilarity score, which will be our fitness value.
    # We want to MAXIMIZE this score.
    fitness = min_based_dissimilarity_score(distance_matrix)

    return fitness


print("✅ Fitness function defined for GA optimization")

## 3. GA Configuration and Execution

We configure the GA parameters and run the optimization to find optimal basis function parameters.


In [ ]:
# # --- Sharing function ---
# def sharing_function(distance, sigma_share, alpha=1):
#     if distance < sigma_share:
#         return 1 - (distance / sigma_share)**alpha
#     else:
#         return 0

In [ ]:
# # Shared Fitness Score
# def fitness_func_with_sharing(ga_instance, chromosome, chromosome_idx):
#     """
#     A wrapper that calls the original fitness function, then applies fitness
#     sharing using your distance_optimal_pairing function.
#     """
#     # Step 1: Get the base fitness score
#     original_fitness = fitness_func(ga_instance, chromosome, chromosome_idx)

#     # Step 2: Calculate the niche count
#     population = ga_instance.population
#     niche_count = 0
#     for chr in population:
#         # *** NOW CALLING YOUR MODIFIED FUNCTION DIRECTLY ***
#         dist = distance_optimal_pairing(chromosome, chr)
#         niche_count += sharing_function(dist, sigma_share=1, alpha=1)

#     # Step 3: Adjust the fitness
#     if niche_count < 1:
#         niche_count = 1
#     adjusted_fitness = original_fitness / niche_count

#     return adjusted_fitness

In [ ]:
# # Custom Mutation Function
# # ==================================================================
# def custom_mutation(offspring, ga_instance):
#     # Loop through each individual in the offspring
#     for chromosome_idx in range(offspring.shape[0]):
#         # Loop through each gene in the chromosome
#         for gene_idx in range(offspring.shape[1]):
#             # Use the GA instance's current mutation probability
#             if np.random.rand() < (ga_instance.mutation_percent_genes / 100.0):

#                 # It's a mu (µ) gene
#                 if gene_idx % 2 == 0:
#                     # Gaussian Perturbation (a small nudge)
#                     noise = np.random.normal(loc=0.0, scale=0.5)
#                     offspring[chromosome_idx, gene_idx] += noise

#                 # It's a sigma (σ) gene
#                 else:
#                     # Multiplicative Perturbation (a small scaling)
#                     noise = 10**np.random.normal(loc=0.0, scale=0.1)
#                     offspring[chromosome_idx, gene_idx] *= noise

#                 # Clamp values to stay within bounds after mutation
#                 offspring[chromosome_idx, gene_idx] = np.clip(
#                     offspring[chromosome_idx, gene_idx],
#                     ga_instance.gene_space[gene_idx]['low'],
#                     ga_instance.gene_space[gene_idx]['high']
#                 )
#     return offspring

In [ ]:
# Function to Generate Distance Matrix (compare 2 sets of basis functions)
# ==================================================================
def generate_distance_matrix(chromosomes_obj_list):
    """
    Generates a pairwise distance matrix for a list of chromosomes_obj.

    Args:
        chromosomes_obj_list (list): A list of Chromosome objects.

    Returns:
        numpy.ndarray: A symmetric matrix of pairwise distances.
    """
    num_chromosomes_obj = len(chromosomes_obj_list)
    distance_matrix = np.zeros((num_chromosomes_obj, num_chromosomes_obj))

    for i in range(num_chromosomes_obj):
        for j in range(i, num_chromosomes_obj):  # Only compute the upper triangle
            if i == j:
                distance_matrix[i, j] = 0
            else:
                dist = distance_optimal_pairing(
                    np.array(chromosomes_obj_list[i].genes), np.array(chromosomes_obj_list[j].genes)
                )
                distance_matrix[i, j] = dist
                distance_matrix[j, i] = dist  # Matrix is symmetric

    return distance_matrix

In [ ]:
# Analyze Chromosome Diversity
# ==================================================================
def analyze_Chromosome_diversity(final_population, top_n, F_threshold, R_peak):
    """
    Analyzes the top N chromosomes in a population to report on the diversity
    and quality of high-performing chromosome families (clusters).

    Args:
        final_population (list): The list of Chromosome objs in the final generation.
        top_n (int): The number of top chromosomes to consider from the population.
        F_threshold (float): The minimum fitness value for a chromosome to be included.
        R_peak (float): The distance radius for clustering.

    Returns:
        dict: A dictionary containing the analysis report.
    """
    # ==================================================================
    # Step 1: Select and filter top, high-quality chromosomes
    # ==================================================================
    final_population.sort(key=lambda x: x.fitness, reverse=True)
    top_chromosomes = final_population[:top_n]
    chr_high = [chr for chr in top_chromosomes if chr.fitness >= F_threshold]

    if not chr_high:
        return {
            "total_chromosomes_analyzed": len(top_chromosomes),
            "high_quality_count": 0,
            "family_count": 0,
            "families": [],
        }

    # ==================================================================
    # Step 2: Cluster chromosomes into families
    # ==================================================================
    chromosome_families = []
    for chr in chr_high:
        found_family = False
        for family in chromosome_families:
            if (
                distance_optimal_pairing(
                    np.array(chr.genes), np.array(family["representative"].genes)
                )
                <= R_peak
            ):
                family["members"].append(chr)
                if chr.fitness > family["top_score"]:
                    family["top_score"] = chr.fitness
                found_family = True
                break

        if not found_family:
            chromosome_families.append(
                {"representative": chr, "members": [chr], "top_score": chr.fitness}
            )

    # ==================================================================
    # Step 3: Prepare the final report
    # ==================================================================
    chromosome_families.sort(key=lambda x: x["top_score"], reverse=True)

    family_reports = []
    for i, family in enumerate(chromosome_families):
        family_reports.append(
            {
                "family_id": i + 1,
                "top_score": family["top_score"],
                "member_count": len(family["members"]),
                "representative_chromosome": family["representative"].genes,
            }
        )

    report = {
        "total_chromosomes_analyzed": len(top_chromosomes),
        "high_quality_count": len(chr_high),
        "family_count": len(chromosome_families),
        "families": family_reports,
        "high_quality_chromosomes": chr_high,
    }

    return report

In [ ]:
# def vat_reorder


def vat_reorder(distance_matrix):
    """
    Reorders a given distance matrix using the VAT method.

    Parameters:
    - distance_matrix (2D ndarray): Pairwise distance matrix.

    Returns:
    - vat_matrix (2D ndarray): Reordered distance matrix for cluster visualization.
    - reorder (list): The reordering of indices applied to the distance matrix.
    """
    n = distance_matrix.shape[0]

    # Step 1: Find the most dissimilar pair (farthest apart)
    i, j = np.unravel_index(np.argmax(distance_matrix, axis=None), distance_matrix.shape)

    # Step 2: Start with one of the most distant points
    reorder = [i]
    remaining = set(range(n)) - {i}  # Remaining points to process

    # Step 3: Iteratively find the next closest point to any selected point
    while remaining:
        next_index = min(remaining, key=lambda x: min(distance_matrix[x, p] for p in reorder))
        reorder.append(next_index)
        remaining.remove(next_index)

    # Step 4: Apply the reordering to the distance matrix
    vat_matrix = distance_matrix[np.ix_(reorder, reorder)]

    return vat_matrix, reorder

In [ ]:
def iVAT_transform(vat_matrix):
    """
    Compute the iVAT distance transform (path-based distances) from a VAT-reordered matrix.

    The input vat_matrix should already be reordered via VAT, meaning its rows and columns
    follow a minimum spanning tree-like sequence. This function replaces each direct distance
    with the “bottleneck” distance along the MST path—i.e., the maximum edge on that path.

    Algorithm steps:
    For each new row r (1..N-1):
        - Find j < r that yields the minimum distance in vat_matrix[r, :r].
          (This j is the MST edge that would connect r to the existing tree.)
        - Set iVat[r, j] to that distance (symmetric as well).
        - For each c < r, c != j:
          iVat[r, c] = max( vat_matrix[r, j], iVat[j, c] )
          (The path from r to c goes through j, so the cost is the maximum of
          the r->j edge and the already computed j->c path.)

    Parameters
    ----------
    vat_matrix : ndarray of shape (N, N)
        The VAT-reordered distance matrix, typically symmetrical and
        with zeros on the diagonal.

    Returns
    -------
    ivat_matrix : ndarray of shape (N, N)
        The iVAT-transformed distance matrix in the same VAT ordering.
    """
    N = vat_matrix.shape[0]
    ivat_matrix = np.zeros_like(vat_matrix)

    for r in range(1, N):
        # Find the closest connection j for row r among previously processed rows
        j = np.argmin(vat_matrix[r, :r])

        # Direct MST edge between r and j
        ivat_matrix[r, j] = vat_matrix[r, j]
        ivat_matrix[j, r] = ivat_matrix[r, j]

        # Update path-based distances to all other previous vertices c
        for c in range(r):
            if c != j:
                # Bottleneck distance: the worst edge on the path r->j->c
                ivat_matrix[r, c] = max(vat_matrix[r, j], ivat_matrix[j, c])
                ivat_matrix[c, r] = ivat_matrix[r, c]

    return ivat_matrix

In [ ]:
# Visualization of Distance Matrix
# ==================================================================


def visualize_distance_matrix_large(
    distance_matrix, title="Distance Matrix Visualization", figure_size=(8, 6)
):
    """
    Visualizes a distance matrix as a heatmap.

    Parameters:
    - distance_matrix (2D ndarray): Pairwise distance matrix.
    - title (str): Title for the plot.
    """
    plt.figure(figsize=figure_size)
    plt.imshow(distance_matrix, cmap="viridis", origin="upper")
    plt.colorbar(label="Distance")
    plt.title(title)
    plt.xlabel("Indices")
    plt.ylabel("Indices")
    plt.tight_layout()
    plt.show()

In [ ]:
# Run GA
# ==================================================================
# Section 3: GA Configuration
# ==================================================================
# --- Define parameter bounds ---
MU_MIN, MU_MAX = 4.0, 20.0
SIGMA_MIN, SIGMA_MAX = 0.1, 4.0

# --- Define the gene space for PyGAD's initialization ---
# Gene space is a list of dictionaries defining the range for each of the 8 genes.
# Genes 0, 2, 4, 6 are means (µ)
# Genes 1, 3, 5, 7 are standard deviations (σ)
gene_space = []
for i in range(4):
    gene_space.append({"low": MU_MIN, "high": MU_MAX})  # Space for µ
    gene_space.append({"low": SIGMA_MIN, "high": SIGMA_MAX})  # Space for σ

# --- GA Parameters ---
POPULATION_SIZE = 200
NUM_GENERATIONS = 1000
NUM_PARENTS_MATING = 50  # Number of chromosomes to be selected as parents
NUM_GENES = 8  # 4 pairs of (µ, σ)

# --- GA Operator Settings moved to the top for easy configuration ---
PARENT_SELECTION_TYPE = "sus"
# K_TOURNAMENT = 3 # The size of the tournament for parent selection
CROSSOVER_TYPE = "uniform"
MUTATION_TYPE = "adaptive"
# MUTATION_PERCENT_GENES = 10
MUTATION_PROBABILITY = [0.5, 0.3]
KEEP_ELITISM = 5
RANDOM_SEED = 42

# --- Hall of Fame & Generational Tracking Setup ---
HALL_OF_FAME_SIZE = 100
hall_of_fame = []
generational_bests = []

# --- Lists to store fitness statistics per generation ---
best_fitness_per_gen = []
worst_fitness_per_gen = []
mean_fitness_per_gen = []

# --- Dictionary to store fitness scores at intervals ---
fitness_snapshots = {}

# --- Create a progress bar instance (will be used by the callback) ---
pbar = tqdm(total=NUM_GENERATIONS, desc="Evolving Generations")


def on_generation(ga_instance):
    """
    Callback function to perform all tasks after each generation:
    1. Update the Hall of Fame.
    2. Track generational champions.
    3. Update the progress bar.
    """
    # print('test')
    global \
        hall_of_fame, \
        generational_bests, \
        best_fitness_per_gen, \
        worst_fitness_per_gen, \
        mean_fitness_per_gen, \
        fitness_snapshots

    # --- Hall of Fame Logic ---
    population = ga_instance.population
    fitness = ga_instance.last_generation_fitness
    for i in range(len(population)):
        chromosome_tuple = (fitness[i], population[i].copy())
        if len(hall_of_fame) < HALL_OF_FAME_SIZE:
            hall_of_fame.append(chromosome_tuple)
            hall_of_fame.sort(key=lambda x: x[0])
        elif fitness[i] > hall_of_fame[0][0]:
            hall_of_fame[0] = chromosome_tuple
            hall_of_fame.sort(key=lambda x: x[0])

    # --- Generational Champions Logic ---
    gen_num = ga_instance.generations_completed
    if (gen_num % 20 == 0) and (0 < gen_num <= 200):
        chromosome, fitness, _ = ga_instance.best_solution()
        # Using pbar.write to print messages without breaking the bar
        # pbar.write(f"  - Storing champion from Generation {gen_num}")
        generational_bests.append((gen_num, fitness, chromosome.copy()))

    # --- Record Fitness Statistics ---
    last_gen_fitness = ga_instance.last_generation_fitness
    best_fitness_per_gen.append(np.max(last_gen_fitness))
    worst_fitness_per_gen.append(np.min(last_gen_fitness))
    mean_fitness_per_gen.append(np.mean(last_gen_fitness))

    # --- Take a snapshot of the full population's fitness every 500 generations ---
    if (gen_num % 500 == 0) and (gen_num > 0):
        fitness_snapshots[gen_num] = last_gen_fitness.copy()

    # --- Progress Bar Update Logic ---
    pbar.update(1)


# ==================================================================
# Section 4: Run the Genetic Algorithm
# ==================================================================
# print("Running Genetic Algorithm... 🧬")
# fitness_function = make_fitness_func_group(separability_mean_distance_over_variance)  # Use the group-based fitness function
ga_instance = pygad.GA(
    num_generations=NUM_GENERATIONS,
    sol_per_pop=POPULATION_SIZE,
    num_parents_mating=NUM_PARENTS_MATING,
    num_genes=NUM_GENES,
    gene_space=gene_space,
    fitness_func=fitness_func,  # Use the fitness function defined above
    # fitness_func=make_fitness_func_group(separability_mean_distance_over_variance),  # Use the group-based fitness function
    on_generation=on_generation,  # Register the callback function
    # --- Standard Recommended Settings ---
    parent_selection_type=PARENT_SELECTION_TYPE,  # Steady-state selection
    # K_tournament=K_TOURNAMENT,
    crossover_type=CROSSOVER_TYPE,  # Two-points crossover
    mutation_type=MUTATION_TYPE,  # Use random mutation, can be customized
    # mutation_percent_genes=MUTATION_PERCENT_GENES,        # Mutate 15% of the genes in each offspring
    mutation_probability=MUTATION_PROBABILITY,
    # --- Other Settings ---
    keep_elitism=KEEP_ELITISM,  # Keep the 2 best chromosomes from the previous generation
    random_seed=RANDOM_SEED,  # For reproducible results
)
# print('test2')
# --- Run the GA ---
ga_instance.run()
# print('test3')
# --- Close the progress bar after the run is complete ---
pbar.close()
print("GA run complete.")


# # ==================================================================
# # New Section 4: Manual GA Run with Elitism + Deterministic Crowding
# # ==================================================================
# print("Running GA with Elitism + Deterministic Crowding... 🧬")


# ga_instance = pygad.GA(
#     num_generations=NUM_GENERATIONS,
#     sol_per_pop=POPULATION_SIZE,
#     num_parents_mating=NUM_PARENTS_MATING,
#     num_genes=NUM_GENES,
#     gene_space=gene_space,
#     fitness_func=fitness_func,
#     on_generation=on_generation, # Register the callback function

#     # --- Standard Recommended Settings ---
#     parent_selection_type=PARENT_SELECTION_TYPE,      # Steady-state selection
#     # K_tournament=K_TOURNAMENT,
#     crossover_type=CROSSOVER_TYPE,       # Two-points crossover
#     mutation_type=MUTATION_TYPE,           # Use random mutation, can be customized
#     # mutation_percent_genes=MUTATION_PERCENT_GENES,        # Mutate 15% of the genes in each offspring
#     mutation_probability=MUTATION_PROBABILITY,

#     # --- Other Settings ---
#     keep_elitism=KEEP_ELITISM,                   # Keep the 2 best solutions from the previous generation
#     random_seed=RANDOM_SEED                    # For reproducible results
# )

# # --- We need the distance function for crowding ---
# # from scipy.optimize import linear_sum_assignment
# # import numpy as np

# def distance_optimal_pairing(individual_a, individual_b):
#     genes_a = np.array(individual_a).reshape(-1, 2)
#     genes_b = np.array(individual_b).reshape(-1, 2)
#     num_basis = genes_a.shape[0]
#     dist_matrix = np.zeros((num_basis, num_basis))
#     for i in range(num_basis):
#         for j in range(num_basis):
#             dist_matrix[i, j] = np.linalg.norm(genes_a[i] - genes_b[j])
#     rows, cols = linear_sum_assignment(dist_matrix)
#     return dist_matrix[rows, cols].sum()

# # --- NEW: Self-Contained Adaptive Mutation Function ---
# def adaptive_mutation_manual(offspring, fitness_scores, mutation_probs, gene_space):
#     mutated_offspring = offspring.copy()
#     max_prob, min_prob = mutation_probs[0], mutation_probs[1]
#     max_fitness, min_fitness = np.max(fitness_scores), np.min(fitness_scores)

#     if max_fitness == min_fitness: return mutated_offspring

#     for i in range(offspring.shape[0]):
#         fitness = fitness_scores[i]
#         prob = max_prob - (((fitness - min_fitness) / (max_fitness - min_fitness)) * (max_prob - min_prob))

#         for j in range(offspring.shape[1]):
#             if np.random.rand() < prob:
#                 low, high = gene_space[j]['low'], gene_space[j]['high']
#                 mutated_offspring[i, j] = np.random.uniform(low, high)

#     return mutated_offspring


# # ==================================================================
# # THIS IS THE CRITICAL FIX:
# # Calculate fitness for the initial population before starting the loop.
# # ==================================================================
# ga_instance.cal_pop_fitness()

# # --- Manual Generation Loop ---
# for generation in range(NUM_GENERATIONS):
#     # --- 1. Elitism ---
#     current_population = ga_instance.population
#     current_fitness = ga_instance.last_generation_fitness

#     elite_indices = np.argsort(current_fitness)[-KEEP_ELITISM:]
#     elites = current_population[elite_indices]

#     # --- 2. Create Offspring ---
#     parents = current_population
#     offspring_size = (POPULATION_SIZE, NUM_GENES)
#     offspring = ga_instance.crossover(parents=parents, offspring_size=offspring_size)

#     # --- 3. Apply Self-Contained Adaptive Mutation ---
#     unmutated_offspring_fitness = np.array([ga_instance.fitness_func(ga_instance, sol, i) for i, sol in enumerate(offspring)])
#     offspring = adaptive_mutation_manual(offspring, unmutated_offspring_fitness, MUTATION_PROBABILITY, gene_space)
#     offspring_fitness = np.array([ga_instance.fitness_func(ga_instance, sol, i) for i, sol in enumerate(offspring)])

#     # --- 4. Deterministic Crowding ---
#     new_population = np.zeros_like(current_population)
#     new_population[elite_indices] = elites

#     non_elite_indices = list(set(range(POPULATION_SIZE)) - set(elite_indices))
#     np.random.shuffle(non_elite_indices)

#     for i in range(0, len(non_elite_indices), 2):
#         if i + 1 >= len(non_elite_indices):
#             last_idx = non_elite_indices[i]
#             new_population[last_idx] = parents[last_idx]
#             break

#         p1_idx, p2_idx = non_elite_indices[i], non_elite_indices[i+1]
#         p1, p2 = parents[p1_idx], parents[p2_idx]
#         c1, c2 = offspring[p1_idx], offspring[p2_idx]

#         p1_fit, p2_fit = current_fitness[p1_idx], current_fitness[p2_idx]
#         c1_fit, c2_fit = offspring_fitness[p1_idx], offspring_fitness[p2_idx]

#         if distance_optimal_pairing(p1, c1) + distance_optimal_pairing(p2, c2) <= \
#            distance_optimal_pairing(p1, c2) + distance_optimal_pairing(p2, c1):

#             if c1_fit > p1_fit: new_population[p1_idx] = c1
#             else: new_population[p1_idx] = p1

#             if c2_fit > p2_fit: new_population[p2_idx] = c2
#             else: new_population[p2_idx] = p2
#         else:
#             if c2_fit > p1_fit: new_population[p1_idx] = c2
#             else: new_population[p1_idx] = p1

#             if c1_fit > p2_fit: new_population[p2_idx] = c1
#             else: new_population[p2_idx] = p1

#     # --- 5. Update GA Instance ---
#     ga_instance.population = new_population
#     ga_instance.generations_completed = generation + 1
#     ga_instance.cal_pop_fitness()

#     # --- 6. Callback ---
#     on_generation(ga_instance)

# # --- Close the progress bar after the run is complete ---
# p_bar.close()
# print("GA run complete.")

In [ ]:
# --- Retrieve the fitness values of the last generation ---
# This line assumes 'ga_instance' is the variable holding your GA results
last_gen_fitness = ga_instance.last_generation_fitness

# --- Sort the fitness scores in descending order for better visualization ---
sorted_fitness = np.sort(last_gen_fitness)[::-1]

# --- Create the plot ---
plt.figure(figsize=(12, 7))
plt.bar(range(len(sorted_fitness)), sorted_fitness, color="skyblue")
plt.title("Fitness Distribution of the Final Generation", fontsize=16)
plt.xlabel("Individual (Sorted by Fitness)", fontsize=12)
plt.ylabel("Fitness Score", fontsize=12)
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()

In [ ]:
# Post-Run Diversity Analysis

# --- Helper Class to structure the data for analysis ---
# This makes the code cleaner, as our analysis functions expect objects.
class Chromosome:
    def __init__(self, genes, fitness):
        self.genes = genes
        self.fitness = fitness


final_chromosomes = ga_instance.population
final_fitness = ga_instance.last_generation_fitness

final_population_objects = []
for genes, fitness in zip(final_chromosomes, final_fitness):
    final_population_objects.append(Chromosome(genes=genes, fitness=fitness))

F_threshold = 45.0  # Fitness threshold for high-quality chromosomes

analysis_report = analyze_Chromosome_diversity(
    final_population=final_population_objects,
    top_n=100,
    F_threshold=F_threshold,
    R_peak=1,  # This radius might need tuning based on your distance values
)


# --- Step 3: Print the human-readable summary ---
print("\n" + "=" * 40)
print("      Post-Run Diversity Analysis")
print("=" * 40)
print(f"Analysis of Top {analysis_report['total_chromosomes_analyzed']} Chromosomes:")
print(f"Found {analysis_report['high_quality_count']} chromosomes with score >= {F_threshold}.")
print(f"These chromosomes form approximately {analysis_report['family_count']} distinct families.")
print("-" * 40)

if not analysis_report["families"]:
    print("No distinct families found above the threshold.")
else:
    for family in analysis_report["families"]:
        print(f"Chromosome Family {family['family_id']}:")
        print(f"  - Top Score: {family['top_score']:.4f}")
        print(f"  - Members in Top 100: {family['member_count']}")


high_quality_distance_matrix = generate_distance_matrix(analysis_report["high_quality_chromosomes"])

# Compute the iVAT matrix
vat_matrix, reorder = vat_reorder(high_quality_distance_matrix)
ivat_matrix = iVAT_transform(vat_matrix)
# Visualize the original and iVAT matrices
visualize_distance_matrix_large(
    high_quality_distance_matrix, title="Original Distance Matrix", figure_size=(4, 3)
)
visualize_distance_matrix_large(vat_matrix, title="VAT Matrix Visualization", figure_size=(4, 3))
visualize_distance_matrix_large(ivat_matrix, title="iVAT Matrix Visualization", figure_size=(4, 3))


def plot_high_quality_chromosomes(
    high_quality_chromosomes, wavelengths, vertical_offset=0.1, figure_size=(10, 10)
):
    """
    Plots a given list of high-quality chromosomes, typically generated
    from the analysis function.

    Args:
        high_quality_chromosomes (list): A list of Chromosome objects, sorted by fitness.
        wavelengths (np.array): The array of wavelength values for the x-axis.
        vertical_offset (float): The vertical spacing between plots.
        figure_size (tuple): The size of the matplotlib figure.
    """
    # The input is already a sorted list of high-quality Chromosome objects
    num_chromosomes = len(high_quality_chromosomes)
    if num_chromosomes == 0:
        print("No high-quality chromosomes to plot.")
        return

    colors = plt.cm.viridis(np.linspace(0, 1, num_chromosomes))  # Changed colormap for variety
    plt.figure(figsize=figure_size)

    # The list is already sorted descending, so we iterate normally
    # but plot them from bottom to top using their rank.
    for i, chromosome_obj in enumerate(high_quality_chromosomes):
        chromosome = chromosome_obj.genes
        basis_funcs = generate_basis_from_chromosome(chromosome, wavelengths)

        # Plotting from bottom (rank N) to top (rank 1)
        plot_rank_offset = (num_chromosomes - 1 - i) * vertical_offset

        for j, basis_func in enumerate(basis_funcs.T):
            scaled_basis_func = basis_func * 0.4  # Scale for better visualization
            plt.plot(
                wavelengths,
                scaled_basis_func + plot_rank_offset,
                color=colors[i],
                alpha=0.8,
                linewidth=2,
            )

    plt.title(f"Top {num_chromosomes} High-Quality Sensor Designs (Fitness >= Threshold)")
    plt.xlabel("Wavelength (µm)")
    plt.ylabel("Absorptivity (Offset Applied for Visualization)")

    # --- Dynamic Y-axis Ticks ---
    tick_interval = max(1, num_chromosomes // 10)  # Aim for about 10 ticks
    tick_indices = [i for i in range(num_chromosomes) if i % tick_interval == 0]
    if num_chromosomes - 1 not in tick_indices:  # Ensure the last rank is always shown
        tick_indices.append(num_chromosomes - 1)

    rank_labels = [f"Rank {i + 1}" for i in tick_indices]
    rank_offsets = [(num_chromosomes - 1 - i) * vertical_offset for i in tick_indices]

    plt.yticks(rank_offsets, rank_labels)
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.show()


plot_high_quality_chromosomes(
    analysis_report["high_quality_chromosomes"], wavelengths, vertical_offset=0.5
)

In [ ]:
# Section 5: Analyze and Visualize Results
# ==================================================================

# --- Retrieve the best solution ---
chromosome, solution_fitness, chromosome_idx = ga_instance.best_solution()
print("\nBest solution found (µ, σ pairs):")
for i in range(4):
    print(f"  Basis Function {i + 1}: µ = {chromosome[i * 2]:.3f}, σ = {chromosome[i * 2 + 1]:.3f}")
print(f"Fitness value of the best solution = {solution_fitness:.4f}")

# --- Plot Custom Fitness Statistics ---
plt.figure(figsize=(10, 6))
plt.plot(best_fitness_per_gen, label="Best Fitness", color="green")
plt.plot(mean_fitness_per_gen, label="Mean Fitness", color="blue")
plt.plot(worst_fitness_per_gen, label="Worst Fitness", color="red", linestyle="--")
plt.title("GA Fitness Statistics Over Generations", fontsize=16)
plt.xlabel("Generation")
plt.ylabel("Fitness")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

# --- Plot Best Basis Functions "Canvas" ---
best_basis_functions = generate_basis_from_chromosome(chromosome, wavelengths)

plt.figure(figsize=(10, 5))
for i in range(best_basis_functions.shape[1]):
    plt.plot(
        wavelengths,
        best_basis_functions[:, i],
        label=f"µ={chromosome[i * 2]:.2f}, σ={chromosome[i * 2 + 1]:.2f}",
    )

# Plot original substances in the background for context
for i, name in enumerate(substance_names):
    plt.plot(wavelengths, emissivity_curves[:, i], alpha=0.15)

plt.title("Best Basis Functions Found by GA", fontsize=16)
plt.xlabel("Wavelength (µm)")
plt.ylabel("Intensity / Emissivity")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()


# ==================================================================
# Section 11: Plot Fitness Distribution Snapshots
# ==================================================================
plt.figure(figsize=(10, 7))
colors = plt.cm.viridis(np.linspace(0, 1, len(fitness_snapshots)))

# Iterate through the snapshots and plot each one
for i, (gen_num, fitness_scores) in enumerate(fitness_snapshots.items()):
    # Sort the scores for a clean line plot
    sorted_scores = sorted(fitness_scores, reverse=True)
    plt.plot(sorted_scores, color=colors[i], label=f"Generation {gen_num}")

plt.title("Fitness Distribution at Different Generational Snapshots", fontsize=16)
plt.xlabel("Solution Index (Sorted by Fitness)")
plt.ylabel("Fitness Score")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()


# ==================================================================
# Section 9: Plot Top 100 Solutions from the Final Generation
# ==================================================================
def plot_top_100_last_gen(
    final_population, final_fitness, wavelengths, vertical_offset=0.1, figure_size=(10, 40)
):
    """Plots the top 100 solutions from the final population."""

    # Combine population and fitness into a list of tuples
    final_solutions_tuples = list(zip(final_fitness, final_population))

    # Sort by fitness (descending) and take the top 100
    top_solutions = sorted(final_solutions_tuples, key=lambda x: x[0], reverse=True)[:100]

    num_solutions = len(top_solutions)
    if num_solutions == 0:
        print("No solutions in the final generation to plot.")
        return

    colors = plt.cm.cividis(np.linspace(0, 1, num_solutions))
    plt.figure(figsize=figure_size)

    for i in reversed(range(num_solutions)):
        fitness, chromosome = top_solutions[i]
        basis_funcs = generate_basis_from_chromosome(chromosome, wavelengths)
        rank = i
        for j, basis_func in enumerate(basis_funcs.T):
            scaled_basis_func = basis_func * 0.4
            plt.plot(
                wavelengths,
                scaled_basis_func + (num_solutions - 1 - rank) * vertical_offset,
                color=colors[rank],
                alpha=0.8,
                linewidth=2,
            )
    plt.title("Top 100 Sensor Designs from Final Generation")
    plt.xlabel("Wavelength (µm)")
    plt.ylabel("Absorptivity (Offset Applied for Visualization)")

    tick_interval = 10
    rank_labels = [
        f"Rank {i + 1}"
        for i in range(num_solutions)
        if (i % tick_interval == 0) or i == num_solutions - 1
    ]
    rank_offsets = [
        (num_solutions - 1 - i) * vertical_offset
        for i in range(num_solutions)
        if (i % tick_interval == 0) or i == num_solutions - 1
    ]

    if 0 not in [i for i in range(num_solutions) if (i % tick_interval == 0)]:
        rank_labels.append("Rank 1")
        rank_offsets.append((num_solutions - 1) * vertical_offset)

    plt.yticks(rank_offsets, rank_labels)
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.show()


# --- Call the new plotting function ---
plot_top_100_last_gen(
    ga_instance.population, ga_instance.last_generation_fitness, wavelengths, vertical_offset=0.5
)


# # ==================================================================
# # Section 6: Plot Top 10 Solutions from Hall of Fame
# # ==================================================================
# def plot_top_10_from_hof(hall_of_fame, wavelengths, vertical_offset=0.2, figure_size=(10, 6)):
#     """Plots the top 10 solutions from the GA's Hall of Fame."""
#     top_solutions = sorted(hall_of_fame, key=lambda x: x[0], reverse=True)[:10]
#     colors = plt.cm.viridis_r(np.linspace(0, 1, len(top_solutions)))
#     plt.figure(figsize=figure_size)

#     for i in reversed(range(len(top_solutions))):
#         fitness, chromosome = top_solutions[i]
#         basis_funcs = generate_basis_from_chromosome(chromosome, wavelengths)
#         rank = i
#         for j, basis_func in enumerate(basis_funcs.T):
#             scaled_basis_func = basis_func * 0.5
#             plt.plot(
#                 wavelengths,
#                 scaled_basis_func + (len(top_solutions) - 1 - rank) * vertical_offset,
#                 color=colors[rank], alpha=0.8, linewidth=2,
#                 label=f"Rank {rank + 1}" if j == 0 else None
#             )
#     plt.title("Top 10 Sensor Designs Found by GA")
#     plt.xlabel("Wavelength (µm)")
#     plt.ylabel("Absorptivity (Offset Applied for Visualization)")
#     rank_labels = [f"Rank {i+1}" for i in range(len(top_solutions))]
#     rank_offsets = [(len(top_solutions) - 1 - i) * vertical_offset for i in range(len(top_solutions))]
#     plt.yticks(rank_offsets, rank_labels)
#     plt.grid(True, linestyle="--", alpha=0.5)
#     handles, labels = plt.gca().get_legend_handles_labels()
#     plt.legend(handles[::-1], labels[::-1], title="Sensor Ranking", loc="upper left")
#     plt.tight_layout()
#     plt.show()

# if hall_of_fame:
#     plot_top_10_from_hof(hall_of_fame, wavelengths, vertical_offset=0.5)
# else:
#     print("Hall of Fame is empty. Cannot plot top 10.")


# # ==================================================================
# # Section 7: Plot Top 100 Solutions from Hall of Fame
# # ==================================================================
# def plot_top_100_from_hof(hall_of_fame, wavelengths, vertical_offset=0.1, figure_size=(10, 40)):
#     """Plots the top 100 solutions from the GA's Hall of Fame."""
#     top_solutions = sorted(hall_of_fame, key=lambda x: x[0], reverse=True)[:100]
#     num_solutions = len(top_solutions)
#     colors = plt.cm.viridis_r(np.linspace(0, 1, num_solutions))
#     plt.figure(figsize=figure_size)

#     for i in reversed(range(num_solutions)):
#         fitness, chromosome = top_solutions[i]
#         basis_funcs = generate_basis_from_chromosome(chromosome, wavelengths)
#         rank = i
#         for j, basis_func in enumerate(basis_funcs.T):
#             scaled_basis_func = basis_func * 0.4
#             plt.plot(
#                 wavelengths,
#                 scaled_basis_func + (num_solutions - 1 - rank) * vertical_offset,
#                 color=colors[rank], alpha=0.8, linewidth=2
#             )
#     plt.title("Top 100 Sensor Designs Found by GA")
#     plt.xlabel("Wavelength (µm)")
#     plt.ylabel("Absorptivity (Offset Applied for Visualization)")
#     tick_interval = 10
#     rank_labels = [f"Rank {i+1}" for i in range(num_solutions) if (i % tick_interval == 0) or i == num_solutions - 1]
#     rank_offsets = [(num_solutions - 1 - i) * vertical_offset for i in range(num_solutions) if (i % tick_interval == 0) or i == num_solutions - 1]
#     if 0 not in [i for i in range(num_solutions) if (i % tick_interval == 0)]:
#         rank_labels.append("Rank 1")
#         rank_offsets.append((num_solutions-1)*vertical_offset)
#     plt.yticks(rank_offsets, rank_labels)
#     plt.grid(True, linestyle="--", alpha=0.5)
#     plt.tight_layout()
#     plt.show()

# if hall_of_fame:
#     plot_top_100_from_hof(hall_of_fame, wavelengths, vertical_offset=0.5)
# else:
#     print("Hall of Fame is empty. Cannot plot top 100.")


# # ==================================================================
# # Section 8: Plot Generational Champions
# # ==================================================================
# def plot_generational_champions(generational_bests, wavelengths, vertical_offset=0.5, figure_size=(10, 12)):
#     """Plots the best solutions found at different generational milestones."""
#     if not generational_bests:
#         print("No generational champions were saved.")
#         return
#     num_solutions = len(generational_bests)
#     colors = plt.cm.viridis_r(np.linspace(0, 1, num_solutions))
#     plt.figure(figsize=figure_size)

#     for i, (gen_num, fitness, chromosome) in enumerate(generational_bests):
#         basis_funcs = generate_basis_from_chromosome(chromosome, wavelengths)
#         for j, basis_func in enumerate(basis_funcs.T):
#             scaled_basis_func = basis_func * 0.5
#             plot_offset = (num_solutions - 1 - i) * vertical_offset
#             plt.plot(
#                 wavelengths,
#                 scaled_basis_func + plot_offset,
#                 color=colors[i], linewidth=2,
#                 label=f"Gen {gen_num}" if j == 0 else None
#             )
#     plt.title("Best Solutions at Early Generational Milestones")
#     plt.xlabel("Wavelength (µm)")
#     plt.ylabel("Absorptivity (Offset Applied for Visualization)")
#     rank_labels = [f"Gen {gb[0]}" for gb in generational_bests]
#     rank_offsets = [(num_solutions - 1 - i) * vertical_offset for i in range(num_solutions)]
#     plt.yticks(rank_offsets, rank_labels)
#     plt.grid(True, linestyle="--", alpha=0.5)
#     handles, labels = plt.gca().get_legend_handles_labels()
#     plt.legend(handles, labels, title="Milestone", loc="upper left")
#     plt.tight_layout()
#     plt.show()

# plot_generational_champions(generational_bests, wavelengths)